## Architecture at a glance

```
                 ┌─────────────────────────────────────────────────────────────┐
                 │                     INGESTION (offline)                       │
                 │                                                               │
   External  ──► │  PDF parse ──► clean ──► structure-aware chunk ──► embed ──►  │ ──► Vector
   clinic PDFs   │  (pypdf)      text      (section + overlap)    (Gemini emb)   │     Index
                 └─────────────────────────────────────────────────────────────┘
                                                                                       │
                 ┌─────────────────────────────────────────────────────────────┐      │
                 │                     QUERY (online)                            │      │
   Clinician ──► │  question ──► embed query ──► retrieve top-k ◄────────────────┼──────┘
   question      │                              (+ metadata filter)             │
                 │                                       │                       │
                 │                                       ▼                       │
                 │              build grounded prompt (context + rules)          │
                 │                                       │                       │
                 │                                       ▼                       │
                 │              Gemini answer  +  inline [citations]             │ ──► Answer
                 │              (refuses if not in context)                      │
                 └─────────────────────────────────────────────────────────────┘
```


In [1]:
!pip install -q google-generativeai pypdf reportlab numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.0 MB/s eta 0:00:00


In [2]:
import google.generativeai as genai

API_KEY = None
# Preferred: Colab secret named GEMINI_API_KEY
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

# Fallback: prompt securely (input is hidden)
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("Paste your Gemini API key: ")

genai.configure(api_key=API_KEY)

EMBED_MODEL = "models/gemini-embedding-001"   # Gemini embeddings
CHAT_MODEL  = "gemini-2.5-flash"            # fast model

# Quick smoke test
_test = genai.GenerativeModel(CHAT_MODEL).generate_content("Reply with the single word: ready")
print("Gemini says:", _test.text.strip())

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Paste your Gemini API key: ··········
Gemini says: ready


In [3]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas

DOCS_DIR = "clinic_docs"
os.makedirs(DOCS_DIR, exist_ok=True)

PATIENTS = [
    {
        "name": "John A. Carter", "mrn": "MRN-100245", "dob": "1958-03-12", "age": 68,
        "note": ("Nephrology Progress Note. CKD Stage 5 on in-center hemodialysis 3x/week (Mon/Wed/Fri). "
                 "Primary etiology: diabetic nephropathy. Dialysis access: left upper-arm AV fistula, "
                 "functioning well, no signs of stenosis. Interdialytic weight gain averaging 2.4 kg. "
                 "Reports mild fatigue and intermittent leg cramps during the last hour of sessions. "
                 "Blood pressure pre-dialysis 152/88. Adequacy: most recent Kt/V 1.5 (target >=1.4)."),
        "labs": [("Hemoglobin", "9.8 g/dL", "Low"), ("Potassium", "5.6 mEq/L", "High"),
                 ("Phosphorus", "6.1 mg/dL", "High"), ("Calcium", "8.9 mg/dL", "Normal"),
                 ("Albumin", "3.6 g/dL", "Low-normal"), ("Kt/V", "1.5", "Adequate")],
        "meds": ["Epoetin alfa 4000 units IV with dialysis", "Sevelamer 800 mg PO TID with meals",
                 "Calcitriol 0.25 mcg PO daily", "Lisinopril 10 mg PO daily", "Insulin glargine 22 units SC nightly"],
    },
    {
        "name": "Maria L. Gonzalez", "mrn": "MRN-100871", "dob": "1971-09-30", "age": 54,
        "note": ("Outpatient Nephrology Visit. CKD Stage 4 (eGFR 22), not yet on dialysis. "
                 "Etiology: hypertensive nephrosclerosis. Pre-emptive transplant evaluation in progress. "
                 "Patient counseled on AV fistula placement vs peritoneal dialysis if function declines. "
                 "Volume status euvolemic. BP 138/84 on current regimen. No edema. Reports good adherence."),
        "labs": [("eGFR", "22 mL/min/1.73m2", "Low"), ("Creatinine", "2.8 mg/dL", "High"),
                 ("Potassium", "4.9 mEq/L", "Normal"), ("Bicarbonate", "20 mEq/L", "Low"),
                 ("Hemoglobin", "11.2 g/dL", "Low-normal"), ("Urine ACR", "640 mg/g", "High")],
        "meds": ["Losartan 100 mg PO daily", "Amlodipine 5 mg PO daily",
                 "Sodium bicarbonate 650 mg PO BID", "Furosemide 20 mg PO daily as needed"],
    },
    {
        "name": "Robert T. Nguyen", "mrn": "MRN-101533", "dob": "1949-06-05", "age": 76,
        "note": ("Home Peritoneal Dialysis Follow-up. ESRD on automated peritoneal dialysis (APD) nightly. "
                 "Etiology: polycystic kidney disease. PD catheter exit site clean, no erythema or discharge. "
                 "One episode of cloudy effluent two weeks ago treated empirically for peritonitis with "
                 "intraperitoneal antibiotics; culture grew coagulase-negative Staphylococcus, now resolved. "
                 "Residual renal function preserved. Reports good energy. BP 128/76."),
        "labs": [("Hemoglobin", "10.4 g/dL", "Low"), ("Potassium", "4.2 mEq/L", "Normal"),
                 ("Phosphorus", "4.8 mg/dL", "Slightly high"), ("Albumin", "3.9 g/dL", "Normal"),
                 ("PD Kt/V (weekly)", "1.8", "Adequate"), ("CRP", "3.1 mg/L", "Normal")],
        "meds": ["Darbepoetin alfa 40 mcg SC every 2 weeks", "Lanthanum carbonate 500 mg PO TID",
                 "Cinacalcet 30 mg PO daily", "Aspirin 81 mg PO daily"],
    },
]

def make_pdf(p):
    path = os.path.join(DOCS_DIR, p["mrn"].replace("-", "_") + ".pdf")
    c = canvas.Canvas(path, pagesize=letter)
    width, height = letter
    y = height - inch

    def line(txt, dy=16, font="Helvetica", size=11):
        nonlocal y
        c.setFont(font, size)
        c.drawString(inch, y, txt)
        y -= dy

    line("RIVER VALLEY NEPHROLOGY ASSOCIATES  -  EXTERNAL CLINIC RECORD", 22, "Helvetica-Bold", 12)
    line(f"Patient: {p['name']}    MRN: {p['mrn']}", 16, "Helvetica-Bold", 11)
    line(f"DOB: {p['dob']}    Age: {p['age']}", 20)

    line("CLINICAL NOTE", 18, "Helvetica-Bold", 11)
    # wrap the note
    words, cur = p["note"].split(), ""
    for w in words:
        if len(cur) + len(w) + 1 > 95:
            line(cur); cur = w
        else:
            cur = (cur + " " + w).strip()
    if cur:
        line(cur)
    y -= 6

    line("LABORATORY RESULTS", 18, "Helvetica-Bold", 11)
    for name, val, flag in p["labs"]:
        line(f"  - {name}: {val}  ({flag})", 14)
    y -= 6

    line("CURRENT MEDICATIONS", 18, "Helvetica-Bold", 11)
    for m in p["meds"]:
        line(f"  - {m}", 14)

    c.showPage()
    c.save()
    return path

paths = [make_pdf(p) for p in PATIENTS]
print("Generated", len(paths), "synthetic patient PDFs:")
for pth in paths:
    print("  ", pth, "-", os.path.getsize(pth), "bytes")


Generated 3 synthetic patient PDFs:
   clinic_docs/MRN_100245.pdf - 2421 bytes
   clinic_docs/MRN_100871.pdf - 2307 bytes
   clinic_docs/MRN_101533.pdf - 2409 bytes


In [4]:
from pypdf import PdfReader

def load_pdfs(folder):
    docs = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        reader = PdfReader(os.path.join(folder, fname))
        for page_num, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or "").strip()
            if text:
                docs.append({"source": fname, "page": page_num, "text": text})
    return docs

raw_docs = load_pdfs(DOCS_DIR)
print(f"Loaded {len(raw_docs)} pages from {len(set(d['source'] for d in raw_docs))} documents.\n")
print("--- Sample extracted page ---")
print(raw_docs[0]["text"][:500], "...")


Loaded 3 pages from 3 documents.

--- Sample extracted page ---
RIVER VALLEY NEPHROLOGY ASSOCIATES  -  EXTERNAL CLINIC RECORD
Patient: John A. Carter    MRN: MRN-100245
DOB: 1958-03-12    Age: 68
CLINICAL NOTE
Nephrology Progress Note. CKD Stage 5 on in-center hemodialysis 3x/week (Mon/Wed/Fri). Primary
etiology: diabetic nephropathy. Dialysis access: left upper-arm AV fistula, functioning well,
no signs of stenosis. Interdialytic weight gain averaging 2.4 kg. Reports mild fatigue and
intermittent leg cramps during the last hour of sessions. Blood pressure p ...


In [5]:
import re

SECTION_HEADERS = ["CLINICAL NOTE", "LABORATORY RESULTS", "CURRENT MEDICATIONS"]

def split_sections(text):
    """Split a page into (section_name, section_text) using known headers."""
    # Build a regex that finds any header
    pattern = "(" + "|".join(re.escape(h) for h in SECTION_HEADERS) + ")"
    parts = re.split(pattern, text)
    sections, current = [], ("HEADER", parts[0])
    i = 1
    while i < len(parts):
        name = parts[i]
        body = parts[i + 1] if i + 1 < len(parts) else ""
        sections.append((name, body.strip()))
        i += 2
    if parts[0].strip():
        sections.insert(0, current)
    return sections

def extract_mrn(text):
    m = re.search(r"MRN[-:\s]*([0-9]{6})", text)
    return f"MRN-{m.group(1)}" if m else "UNKNOWN"

def word_chunks(text, max_words=120, overlap=25):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    out, start = [], 0
    while start < len(words):
        out.append(" ".join(words[start:start + max_words]))
        start += max_words - overlap
    return out

def build_chunks(docs):
    chunks = []
    for d in docs:
        mrn = extract_mrn(d["text"])
        for sect_name, sect_text in split_sections(d["text"]):
            if not sect_text:
                continue
            for piece in word_chunks(sect_text):
                chunks.append({
                    "id": len(chunks),
                    "text": piece,
                    "source": d["source"],
                    "page": d["page"],
                    "patient_mrn": mrn,
                    "section": sect_name,
                })
    return chunks

chunks = build_chunks(raw_docs)
print(f"Built {len(chunks)} chunks.\n")
for c in chunks[:4]:
    print(f"[id {c['id']}] {c['patient_mrn']} | {c['section']} | {c['source']} p{c['page']}")
    print("   ", c["text"][:120], "...\n")


Built 12 chunks.

[id 0] MRN-100245 | HEADER | MRN_100245.pdf p1
    RIVER VALLEY NEPHROLOGY ASSOCIATES  -  EXTERNAL CLINIC RECORD
Patient: John A. Carter    MRN: MRN-100245
DOB: 1958-03-12 ...

[id 1] MRN-100245 | CLINICAL NOTE | MRN_100245.pdf p1
    Nephrology Progress Note. CKD Stage 5 on in-center hemodialysis 3x/week (Mon/Wed/Fri). Primary
etiology: diabetic nephro ...

[id 2] MRN-100245 | LABORATORY RESULTS | MRN_100245.pdf p1
    - Hemoglobin: 9.8 g/dL  (Low)
  - Potassium: 5.6 mEq/L  (High)
  - Phosphorus: 6.1 mg/dL  (High)
  - Calcium: 8.9 mg/dL  ...

[id 3] MRN-100245 | CURRENT MEDICATIONS | MRN_100245.pdf p1
    - Epoetin alfa 4000 units IV with dialysis
  - Sevelamer 800 mg PO TID with meals
  - Calcitriol 0.25 mcg PO daily
  - L ...



In [6]:
import numpy as np
import time

def embed_texts(texts, task_type, batch_size=50):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = genai.embed_content(model=EMBED_MODEL, content=batch, task_type=task_type)
        vecs.extend(resp["embedding"])
    return np.array(vecs, dtype="float32")

chunk_texts = [c["text"] for c in chunks]
chunk_vecs = embed_texts(chunk_texts, task_type="retrieval_document")

# L2-normalize so dot product == cosine similarity
chunk_vecs /= (np.linalg.norm(chunk_vecs, axis=1, keepdims=True) + 1e-10)

print("Embedded", chunk_vecs.shape[0], "chunks into dimension", chunk_vecs.shape[1])

Embedded 12 chunks into dimension 3072


In [7]:
def retrieve(query, k=4, patient_mrn=None):
    qv = embed_texts([query], task_type="retrieval_query")[0]
    qv /= (np.linalg.norm(qv) + 1e-10)

    if patient_mrn:
        idxs = [i for i, c in enumerate(chunks) if c["patient_mrn"] == patient_mrn]
    else:
        idxs = list(range(len(chunks)))

    sims = chunk_vecs[idxs] @ qv
    order = np.argsort(-sims)[:k]
    return [(chunks[idxs[o]], float(sims[o])) for o in order]

# quick look
for c, s in retrieve("What is the patient's dialysis access?", k=3):
    print(f"{s:.3f}  {c['patient_mrn']} | {c['section']} | {c['source']} p{c['page']}")


0.750  MRN-100245 | CLINICAL NOTE | MRN_100245.pdf p1
0.725  MRN-101533 | CLINICAL NOTE | MRN_101533.pdf p1
0.720  MRN-100871 | CLINICAL NOTE | MRN_100871.pdf p1


In [8]:
SYSTEM_RULES = """You are a clinical documentation assistant for kidney care.
Answer ONLY using the CONTEXT below. Rules:
1. After every factual claim, cite the source as [filename, p.PAGE].
2. If the answer is not in the context, reply exactly: "I don't have that information in the provided records."
3. Never guess lab values, doses, or diagnoses. Be concise and clinical.
"""

def format_context(hits):
    blocks = []
    for c, score in hits:
        blocks.append(f"[{c['source']}, p.{c['page']}] (patient {c['patient_mrn']}, {c['section']})\n{c['text']}")
    return "\n\n".join(blocks)

def answer(query, k=4, patient_mrn=None, show_sources=True):
    hits = retrieve(query, k=k, patient_mrn=patient_mrn)
    context = format_context(hits)
    prompt = f"{SYSTEM_RULES}\n\nCONTEXT:\n{context}\n\nQUESTION: {query}\n\nANSWER:"
    resp = genai.GenerativeModel(CHAT_MODEL).generate_content(prompt)
    out = resp.text.strip()
    print("Q:", query)
    print("\nA:", out)
    if show_sources:
        print("\nRetrieved from:")
        for c, s in hits:
            print(f"   - {c['source']} p{c['page']} ({c['section']}, sim={s:.2f})")
    print("-" * 80)
    return out, hits


Testing!

In [9]:
_ = answer("What dialysis access does John Carter have and how is it functioning?",
           patient_mrn="MRN-100245")

_ = answer("Summarize Maria Gonzalez's kidney function and current CKD stage.",
           patient_mrn="MRN-100871")

# Out-of-scope question -> should refuse, not hallucinate
_ = answer("What is Robert Nguyen's most recent HbA1c?",
           patient_mrn="MRN-101533")

# Cross-patient question with no filter (system retrieves across all charts)
_ = answer("Which patient is on home peritoneal dialysis and why?")


Q: What dialysis access does John Carter have and how is it functioning?

A: John A. Carter has a left upper-arm AV fistula for dialysis access [MRN_100245.pdf, p.1]. It is functioning well with no signs of stenosis [MRN_100245.pdf, p.1].

Retrieved from:
   - MRN_100245.pdf p1 (HEADER, sim=0.71)
   - MRN_100245.pdf p1 (CLINICAL NOTE, sim=0.70)
   - MRN_100245.pdf p1 (LABORATORY RESULTS, sim=0.67)
   - MRN_100245.pdf p1 (CURRENT MEDICATIONS, sim=0.66)
--------------------------------------------------------------------------------
Q: Summarize Maria Gonzalez's kidney function and current CKD stage.

A: Maria L. Gonzalez, MRN-100871, has an eGFR of 22 mL/min/1.73m2 [MRN_100871.pdf, p.1]. Her creatinine is 2.8 mg/dL [MRN_100871.pdf, p.1]. She is diagnosed with CKD Stage 4 [MRN_100871.pdf, p.1].

Retrieved from:
   - MRN_100871.pdf p1 (HEADER, sim=0.74)
   - MRN_100871.pdf p1 (LABORATORY RESULTS, sim=0.74)
   - MRN_100871.pdf p1 (CLINICAL NOTE, sim=0.72)
   - MRN_100871.pdf p1 (CURRENT ME

Evaluation:

1. Retrieval hit rate
2. Answer faithfulness

## RAG System User Interface

In [10]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create widgets
question_input = widgets.Textarea(
    value='',
    placeholder='Type your question here...',
    description='Question:',
    disabled=False,
    layout=widgets.Layout(width='auto', height='80px')
)

patient_mrn_input = widgets.Text(
    value='',
    placeholder='e.g., MRN-100245 (optional)',
    description='Patient MRN:',
    disabled=False,
    layout=widgets.Layout(width='auto')
)

submit_button = widgets.Button(
    description='Get Answer',
    disabled=False,
    button_style='info',
    tooltip='Click to get the answer',
    icon='check'
)

output_area = widgets.Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        query = question_input.value.strip()
        patient_mrn = patient_mrn_input.value.strip()

        if not query:
            print("Please enter a question.")
            return

        if patient_mrn and not patient_mrn.startswith('MRN-'):
            print("Invalid MRN format. Please use 'MRN-XXXXXX'.")
            return

        ans, hits = answer(query, patient_mrn=patient_mrn if patient_mrn else None, show_sources=True)

# Attach event listener
submit_button.on_click(on_button_click)

# Display the UI
display(
    widgets.VBox([
        question_input,
        patient_mrn_input,
        submit_button,
        output_area
    ])
)

demo sequence:
- John Carter → *"What is his dialysis access and Kt/V?"* → grounded answer with citations.
- Robert Nguyen → *"What is his most recent HbA1c?"* → it **refuses** (not in the records).
- All patients → *"Which patient is on home peritoneal dialysis?"* → cross-chart retrieval.

Evaluation Check!


In [11]:
EVAL = [
    {"q": "What is John Carter's dialysis schedule?",        "patient": "MRN-100245", "expect_source": "MRN_100245.pdf"},
    {"q": "What is Maria Gonzalez's eGFR?",                  "patient": "MRN-100871", "expect_source": "MRN_100871.pdf"},
    {"q": "What antibiotic situation did Robert Nguyen have?","patient": "MRN-101533", "expect_source": "MRN_101533.pdf"},
    {"q": "What phosphate binder is John Carter taking?",    "patient": "MRN-100245", "expect_source": "MRN_100245.pdf"},
]

def judge_faithful(question, answer_text, context):
    j_prompt = (
        "You are grading a clinical answer. Reply with only YES or NO.\n"
        "Is EVERY factual claim in the ANSWER directly supported by the CONTEXT?\n\n"
        f"CONTEXT:\n{context}\n\nANSWER:\n{answer_text}\n\nSupported (YES/NO):"
    )
    r = genai.GenerativeModel(CHAT_MODEL).generate_content(j_prompt)
    return r.text.strip().upper().startswith("YES")

hits_count, faithful_count = 0, 0
for ex in EVAL:
    hits = retrieve(ex["q"], k=4, patient_mrn=ex["patient"])
    sources = [h[0]["source"] for h in hits]
    hit = ex["expect_source"] in sources
    hits_count += hit

    context = format_context(hits)
    ans = genai.GenerativeModel(CHAT_MODEL).generate_content(
        f"{SYSTEM_RULES}\n\nCONTEXT:\n{context}\n\nQUESTION: {ex['q']}\n\nANSWER:").text.strip()
    faithful = judge_faithful(ex["q"], ans, context)
    faithful_count += faithful

    print(f"Q: {ex['q']}")
    print(f"   retrieval hit: {hit} | faithful: {faithful}")

n = len(EVAL)
print("\n=== RESULTS ===")
print(f"Retrieval hit rate : {hits_count}/{n} = {hits_count/n:.0%}")
print(f"Answer faithfulness: {faithful_count}/{n} = {faithful_count/n:.0%}")


Q: What is John Carter's dialysis schedule?
   retrieval hit: True | faithful: True
Q: What is Maria Gonzalez's eGFR?
   retrieval hit: True | faithful: True
Q: What antibiotic situation did Robert Nguyen have?
   retrieval hit: True | faithful: True
Q: What phosphate binder is John Carter taking?
   retrieval hit: True | faithful: True

=== RESULTS ===
Retrieval hit rate : 4/4 = 100%
Answer faithfulness: 4/4 = 100%
